## tl;dr
This audit executed all code cells with 14 current-build brackets. 0 direct-band frequencies remain unmeasured. The result is diagnostic evidence only.


## Context & Methods
GPIO20 conducted TONE comparison on wspr5, 2026-08-30. Each observation subtracts a linearly interpolated GPSDO reference from the RP1 carrier. Both sources are never intentionally enabled together.

### Key Assumptions
The GPSDO is locked; receiver drift between the references is approximately linear. These are single-bracket diagnostics, not a calibration constant or spectral qualification. Earlier module results are retained but never pooled with the current build.

## Data
The companion JSON retains source capture paths, hashes, clock-monitoring values, and individual estimator outputs. This notebook performs no hardware or network operations.

In [1]:
import json, math
from pathlib import Path
data_path = globals().get('DATA_PATH', Path('results.json'))
data = json.loads(data_path.read_text())
current_build = [r for r in data['measurements'] if r['module_sha256'] == data['current_module_sha256']]
restarted = [r for r in current_build if r.get('campaign') == 'post_usb_reset_paced']
current = restarted if restarted else current_build
missing = sorted(set(data['expected_frequencies_hz']) - {r['frequency_hz'] for r in current})
print('Current-build brackets:', len(current))
print('Unmeasured direct frequencies:', missing)


Current-build brackets: 14
Unmeasured direct frequencies: []


### Validate calculations
Recompute reference interpolation and ppm independently from the saved inputs.

In [2]:
for row in data['measurements']:
    estimate = row['analysis']
    first, tx, last = [estimate[k] for k in ('gps_before','rp1','gps_after')]
    weight = (tx['mid_s']-first['mid_s'])/(last['mid_s']-first['mid_s'])
    reference = (1-weight)*first['residual_hz']+weight*last['residual_hz']
    ppm = (tx['residual_hz']-reference)/row['frequency_hz']*1e6
    assert math.isclose(ppm,row['error_ppm'],abs_tol=1e-9)
    for state in row['chrony'].values():
        assert abs(state['Residual freq']) <= .5 and state['Skew'] <= .5
    assert row['clock_during']['parent_rate'] == 200000000
    assert row['clock_before'] == row['clock_after']
print('Saved calculations and numeric Chrony gates verified.')


Saved calculations and numeric Chrony gates verified.


## Results
Restarted paced sweep only when present; earlier attempts and previous-build observations remain separately identified in the JSON.

In [3]:
print('Frequency Hz | error ppm | FFT cross-check ppm | reference drift Hz')
for row in sorted(current,key=lambda r:r['frequency_hz']):
    print(f"{row['frequency_hz']:.0f} | {row['error_ppm']:.4f} | {row['fft_error_ppm']:.4f} | {row['reference_drift_hz']:.5f}")


Frequency Hz | error ppm | FFT cross-check ppm | reference drift Hz
137500 | -46.2674 | -46.2686 | -0.00106
475700 | -46.2578 | -46.2587 | -0.00238
1838100 | -46.2591 | -46.2571 | -0.00924
3570100 | -46.2637 | -46.2610 | -0.01502
5288700 | -46.2599 | -46.2604 | -0.01793
7040100 | -46.2604 | -46.2598 | -0.01485
10140200 | -46.2535 | -46.2540 | -0.03824
14097100 | -46.2502 | -46.2502 | -0.04772
18106100 | -46.2515 | -46.2514 | -0.07532
21096100 | -46.2465 | -46.2473 | -0.11160
24926100 | -46.2511 | -46.2496 | -0.09190
28126100 | -46.2462 | -46.2463 | -0.09920
50294500 | -46.1905 | -46.1902 | -0.28464
70092500 | -46.1747 | -46.1747 | -0.37203


## Takeaways
Do not infer a calibration constant from a single bracket per frequency. 2m remains untested without a harmonic implementation. The 80m attempt on the preceding module failed closed during parent selection and exposed an uncaught application error; the retry repair is separately identified. Longer finite-TONE behavior also remains unresolved.

### VHF window sensitivity
Alternative reference and tone windows test sensitivity, not a formal uncertainty budget. Comparative temporal stability remains a separate Issue 429 closeout requirement; see stability-comparison-plan.md.

In [4]:
sensitivity = json.loads(data_path.with_name('window-sensitivity.json').read_text())
for row in sensitivity:
    change = abs(row['central_tone_near_reference_ppm'] - row['full_reference_ppm'])
    assert change < .004
    print(f"{row['frequency_hz']} Hz: window sensitivity {change:.6f} ppm")


50294500 Hz: window sensitivity 0.003291 ppm
70092500 Hz: window sensitivity 0.003769 ppm
